# AI Agents - Hands-On Tour

A single notebook walking through `../docs/`: the basic reason-act-
observe loop &rarr; a real, verified failure and its fix &rarr; stopping
conditions &rarr; guardrails &rarr; tracing &rarr; evaluation. Everything runs
against a **local Ollama model** (`llama3.1:8b`) - no hosted API, no
signup.

This module's central lesson came from actually running code, not
from theory: a correctly-built agent loop, allowed to batch multiple
tool calls into one turn, restarted a service under a condition that
wasn't met. This notebook reproduces that finding directly.

## Setup

```bash
ollama pull llama3.1:8b
pip install openai
```

In [ ]:
import json
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "llama3.1:8b"

DISK_USAGE = {"db-primary-01": 92, "web-node-01": 41}

def get_disk_usage(hostname: str) -> dict:
    return {"hostname": hostname, "disk_percent": DISK_USAGE.get(hostname, 50)}

def restart_service(service_name: str) -> dict:
    return {"service": service_name, "status": "restarted"}

TOOLS = {"get_disk_usage": get_disk_usage, "restart_service": restart_service}

TOOL_SCHEMAS = [
    {"type": "function", "function": {"name": "get_disk_usage",
        "description": "Get current disk usage percentage for a host.",
        "parameters": {"type": "object", "properties": {"hostname": {"type": "string"}}, "required": ["hostname"]}}},
    {"type": "function", "function": {"name": "restart_service",
        "description": "Restart a named service. Only call this if disk usage was confirmed ABOVE 90 percent by a previous get_disk_usage call.",
        "parameters": {"type": "object", "properties": {"service_name": {"type": "string"}}, "required": ["service_name"]}}},
]

## 1. The basic agent loop

See `docs/07-Building-a-Minimal-Agent-From-Scratch.md`. Allows the
model to request multiple tool calls in one turn.

In [ ]:
def run_agent_broken(goal, max_steps=5):
    messages = [
        {"role": "system", "content": "You are an ops agent. Use tools step by step."},
        {"role": "user", "content": goal},
    ]
    tools_called = []
    for _ in range(max_steps):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0)
        msg = response.choices[0].message
        if msg.tool_calls:
            messages.append(msg)
            for call in msg.tool_calls:
                args = json.loads(call.function.arguments)
                tools_called.append(call.function.name)
                result = TOOLS[call.function.name](**args)
                messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
        else:
            return msg.content, tools_called
    return "max steps reached", tools_called

answer, tools_called = run_agent_broken("Check disk usage on db-primary-01, and if it's above 90%, restart the cleanup-service.")
print("tools called:", tools_called)
print("answer:", answer)

## 2. The verified failure: batching skips the observe step

See `docs/08-One-Tool-at-a-Time-A-Verified-Reliability-Lesson.md`.
Disk usage on `web-node-01` is 41% - well under the 90% threshold, so
`restart_service` should NOT be called.

In [ ]:
goal = "Check disk usage on web-node-01, and if it is above 90%, restart the cleanup-service."
answer, tools_called = run_agent_broken(goal)
print("tools called:", tools_called)
print("answer:", answer)
if "restart_service" in tools_called:
    print("\n!! INCORRECT: restarted a service despite the condition not being met.")

## 3. The fix: one tool call per turn

Forcing only the FIRST requested tool call per turn - plus a bounded
one-time nudge if the model stops without confirming completion -
fixes it. Verified consistent across repeated runs.

In [ ]:
def run_agent_fixed(goal, max_steps=6):
    messages = [
        {"role": "system", "content": "You are an ops agent. Call ONE tool at a time and wait for its result before deciding the next step."},
        {"role": "user", "content": goal},
    ]
    tools_called = []
    nudged = False
    for _ in range(max_steps):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0)
        msg = response.choices[0].message
        if msg.tool_calls:
            messages.append(msg)
            call = msg.tool_calls[0]  # <- the actual fix
            args = json.loads(call.function.arguments)
            tools_called.append(call.function.name)
            result = TOOLS[call.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
        else:
            if not nudged:
                messages.append({"role": "assistant", "content": msg.content})
                messages.append({"role": "user", "content": "If the goal requires another tool call to be fully complete, make that call now. Otherwise, confirm it is complete."})
                nudged = True
                continue
            return msg.content, tools_called
    return "max steps reached", tools_called

answer, tools_called = run_agent_fixed(goal)
print("tools called:", tools_called)
print("answer:", answer)
if "restart_service" not in tools_called:
    print("\nCorrect: did not restart, since the condition wasn't met.")

## 4. Stopping conditions

See `docs/09-Stopping-Conditions-and-Loop-Limits.md`. A hard step
limit guarantees the loop terminates even on a goal no tool can solve.

In [ ]:
result, _ = run_agent_fixed("Investigate why the sky is blue and fix it.", max_steps=3)
print(result)

## 5. Guardrails: code, not just prompts

See `docs/13-Guardrails-and-Human-in-the-Loop.md`. Applied to this
notebook's exact scenario - a confirmation gate that would have caught
section 2's mistake regardless of how the model reasoned.

In [ ]:
def execute_with_confirmation(name, args, auto_confirm):
    if name == "restart_service" and not auto_confirm:
        return {"error": True, "message": "Action declined by human reviewer."}
    return TOOLS[name](**args)

# simulating a human reviewer who sees the 41% result and declines
result = execute_with_confirmation("restart_service", {"service_name": "cleanup-service"}, auto_confirm=False)
print(result)

## 6. Tracing: making the bug visible

See `docs/14-Observability-Tracing-an-Agents-Steps.md`. Run the
BROKEN loop again, but capture every step so the bug is visible
directly in the data, not just inferred from the final answer.

In [ ]:
def run_agent_traced(goal, max_steps=5):
    trace = []
    messages = [
        {"role": "system", "content": "You are an ops agent. Use tools step by step."},
        {"role": "user", "content": goal},
    ]
    for step in range(max_steps):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0)
        msg = response.choices[0].message
        if msg.tool_calls:
            messages.append(msg)
            for call in msg.tool_calls:
                args = json.loads(call.function.arguments)
                result = TOOLS[call.function.name](**args)
                messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
                trace.append({"step": step, "tool": call.function.name, "args": args, "result": result})
        else:
            return msg.content, trace
    return "max steps reached", trace

_, trace = run_agent_traced(goal)
for entry in trace:
    print(entry)

## 7. Evaluation: score actions, not tone

See `docs/15-Evaluating-Agent-Performance.md`. The broken agent's
final answer sounds reasonable even when it did the wrong thing -
score the actual tool calls instead.

In [ ]:
eval_cases = [
    {"goal": "Check disk usage on db-primary-01, and if it is above 90%, restart the cleanup-service.", "should_restart": True},
    {"goal": "Check disk usage on web-node-01, and if it is above 90%, restart the cleanup-service.", "should_restart": False},
]

def score(run_agent_fn, cases):
    correct = 0
    for case in cases:
        _, tools_called = run_agent_fn(case["goal"])
        if ("restart_service" in tools_called) == case["should_restart"]:
            correct += 1
    return correct / len(cases)

print(f"broken agent score: {score(run_agent_broken, eval_cases):.0%}")
print(f"fixed agent score:  {score(run_agent_fixed, eval_cases):.0%}")

## Wrap-up

| Section | Concept | Docs chapter |
| --- | --- | --- |
| 1 | Basic agent loop | 07 |
| 2 | Verified failure | 08 |
| 3 | The fix | 08 |
| 4 | Stopping conditions | 09 |
| 5 | Guardrails | 13 |
| 6 | Tracing | 14 |
| 7 | Evaluation | 15 |

Not covered here but worth running from `../examples/`:
`08_planning.py` (getting a plan before acting) and
`09_multi_agent_supervisor.py` (routing between two worker agents -
including a real routing bug found and fixed with a few-shot prompt).

Next: `08-Ollama` - running these same local models with more control
over how they're served.